### Silver to Gold: Building BI Ready Tables

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

catalog_name = "ecommerce"

In [ ]:
df_orders = spark.read.table(f"{catalog_name}.silver.slv_orders")

display(df_orders.limit(10))

In [ ]:
# Add gross amount.
gold_orders = df_orders.withColumn("gross_amount", F.col("unit_price") * F.col("quantity"))

# Add discount amount..
gold_orders = gold_orders.withColumn("discount_amount", F.col("discount_pct") * F.col("gross_amount") / 100)

# Add sales amount..
gold_orders = gold_orders.withColumn("sales_amount", F.col("gross_amount") - F.col("discount_amount"))

# Add cupon applied flag..
gold_orders = gold_orders.withColumn("is_coupon_applied", F.when(F.col("coupon_code").isNotNull(), 1).otherwise(0))

# rename dt to date
gold_orders = gold_orders.withColumnRenamed("dt", "date")

# add date id..
gold_orders = gold_orders.withColumn("date_id", F.date_format(F.col("date"), "yyyyMMdd").cast(IntegerType()))

display(gold_orders.limit(10))

In [ ]:
gold_orders.select(["order_id", 'item_seq', 'customer_id', 'product_id', 'unit_price', 'quantity', 'discount_pct', 'gross_amount', 'discount_amount', 'tax_amount', 'sales_amount', 'is_coupon_applied', 'coupon_code', 'channel', "date", "date_id",'order_ts', 'created_at', 'source_file']).write.mode("overwrite").option("mergeSchema",True).saveAsTable(f"{catalog_name}.gold.fact_orders")

In [ ]:
%%sql
SELECT * FROM ecommerce.gold.fact_orders;